# Zero Shot with CLIP on Test Dataset

- Got inspiration for finteuning here: https://github.com/sachinruk/blog/blob/master/_notebooks/2021-03-07-CLIP.ipynb
- Dataset: https://huggingface.co/datasets/AlekseyDorkin/extended_tweet_emojis/tree/main


In [23]:
# you might want to restart the kernel after installation is complete.
!pip install transformers evaluate docarray pillow datasets ipywidgets requests jupyter jupyter_client --upgrade --quiet

## 1. Getting Dataset

In [24]:
from datasets import load_dataset

In [25]:
dataset = load_dataset("AlekseyDorkin/extended_tweet_emojis")

Using custom data configuration AlekseyDorkin--extended_tweet_emojis-4eb99ce06d465da0
Reusing dataset parquet (/root/.cache/huggingface/datasets/AlekseyDorkin___parquet/AlekseyDorkin--extended_tweet_emojis-4eb99ce06d465da0/0.0.0/7328ef7ee03eaf3f86ae40594d46a1cec86161704e02dd19f232d81eee72ade8)


  0%|          | 0/2 [00:00<?, ?it/s]

In [26]:
SEED = 42
train_dataset_full = dataset["train"]
train_test_dataset = train_dataset_full.train_test_split(test_size=0.1, seed=SEED)

Loading cached split indices for dataset at /root/.cache/huggingface/datasets/AlekseyDorkin___parquet/AlekseyDorkin--extended_tweet_emojis-4eb99ce06d465da0/0.0.0/7328ef7ee03eaf3f86ae40594d46a1cec86161704e02dd19f232d81eee72ade8/cache-40036b762878c645.arrow and /root/.cache/huggingface/datasets/AlekseyDorkin___parquet/AlekseyDorkin--extended_tweet_emojis-4eb99ce06d465da0/0.0.0/7328ef7ee03eaf3f86ae40594d46a1cec86161704e02dd19f232d81eee72ade8/cache-907f565d049ab5a6.arrow


In [27]:
train_dataset = train_test_dataset["train"]
test_dataset = train_test_dataset["test"]
val_dataset = dataset["validation"]

In [28]:
from pathlib import Path
import collections
import json

In [29]:
current_dir = !pwd
current_dir = current_dir[0]

def parse_dataset(dataset: object, name: str): 
    image_path_to_caption = collections.defaultdict(list)
    for row in  dataset:
        image_path = str(Path(current_dir, "emojis", f"{row['label']}.png"))
        image_path_to_caption[image_path].append(row["text"].lower().rstrip('.'))

    lines = []
    for image_path, captions in image_path_to_caption.items():
        lines.append(json.dumps({"image_path": image_path, "captions": captions}))

    dataset_path = f"dataset/{name}_dataset.json"
    print(dataset_path)
    with open(dataset_path, "w") as f:
        f.write("\n".join(lines))


In [30]:
parse_dataset(train_dataset, "train")

dataset/train_dataset.json


In [31]:
parse_dataset(val_dataset, "val")

dataset/val_dataset.json


In [32]:
!pip install -r requirements.txt  --quiet
!python run_hybrid_clip.py \
    --output_dir "./hf-repo/emoji-predictor" \
    --text_model_name_or_path="roberta-base" \
    --vision_model_name_or_path="openai/clip-vit-base-patch32" \
    --tokenizer_name="roberta-base" \
    --train_file="./dataset/train_dataset.json" \
    --validation_file="./dataset/val_dataset.json" \
    --do_train --do_eval \
    --num_train_epochs="40" --max_seq_length 96 \
    --per_device_train_batch_size="64" \
    --per_device_eval_batch_size="64" \
    --learning_rate="5e-5" --warmup_steps="0" --weight_decay 0.1 \
    --overwrite_output_dir \
    --preprocessing_num_workers 32 \
    --push_to_hub


× The package index page being used does not have a proper HTML doctype declaration.
╰─> Problematic URL: https://download.pytorch.org/whl/torch_stable.html

note: This is an issue with the page at the URL mentioned above.
hint: You might need to reach out to the owner of that package index, to get this fixed. See https://github.com/pypa/pip/issues/10825 for context.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
docarray 0.13.32 requires rich>=12.0.0, but you have rich 11.2.0 which is incompatible.
Unable to display metrics through TensorBoard because the package is not installed: Please run pip install tensorboard to enable.
07/17/2022 08:50:27 - INFO - absl -   Unable to initialize backend 'tpu_driver': NOT_FOUND: Unable to find driver in registry given worker: 
07/17/2022 08:50:27 - INFO - absl -   Unable to initialize backend 'cuda': module 'jaxlib.xla_e

## 2. Get Emojis as Images

In [ ]:
from docarray import DocumentArray

images = DocumentArray.from_files('emojis/*.png')
images.plot_image_sprites()

In [ ]:
from PIL import Image
emojis_as_images = [Image.open(f"emojis/{i}.png") for i in range(32)]

## 3. Pass Test Dataset Throup CLIP

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch

In [ ]:
predictions = []
references = []
SHARDS = 20
# not fully understanding this, but with torch.no_grad
# our GPU does not run out of memory.
with torch.no_grad():
    for i in range(SHARDS):
        chunk = test_dataset.shard(num_shards=SHARDS, index=i)
        chunk_text = chunk["text"]
        chunk_label = chunk["label"]
        model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to("cuda")
        processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        inputs = processor(text=chunk_text, images=emojis_as_images, return_tensors="pt", padding=True, truncation=True)
        outputs = model(**inputs.to("cuda"))
        # we want the probability for each emoji per sentence.
        logits_per_text = outputs.logits_per_text
        # we take the softmax to get the label probabilities
        probs = logits_per_text.softmax(dim=1)
        predictions_for_chunk = [torch.argmax(prob).item() for prob in probs]
        predictions.extend(predictions_for_chunk)
        references.extend(chunk_label)
        print(f"total predictions so far: {len(predictions)}")
        torch.cuda.empty_cache()

## 4. Evaluate Predictions

In [ ]:
import evaluate
precision_metric = evaluate.load("precision")
# Micro-averaging will put more emphasis on the common classes in the data set. 
# Rare labels shouldn’t influence the overall precision metric heavily.
results = precision_metric.compute(references=references, predictions=predictions, average="micro")
print(results)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

# Create confusion matrix
confusion_mat = confusion_matrix(test_dataset["label"], predictions)

# Visualize confusion matrix
plt.figure(figsize = (20,20))
plt.imshow(confusion_mat, interpolation='nearest', cmap=plt.cm.gray)
plt.title('Confusion matrix')
plt.colorbar()
ticks = np.arange(stop=len(emojis_as_images))
plt.xticks(ticks, ticks)
plt.yticks(ticks, ticks)
plt.ylabel('True labels')
plt.xlabel('Predicted labels')
plt.show()


In [ ]:
import pandas as pd
df_mapping = pd.read_table("mapping.txt", header=None)

In [ ]:
df_mapping[1][1].encode('unicode-escape').decode('ASCII')

In [ ]:
len(df_mapping[1].to_list())